# Week 5 — Decision tree and random forest comparison

Compare a `DecisionTreeRegressor` and `RandomForestRegressor` with the Week 4 `LinearRegression` baseline on **the same held-out sales**. The input is `csv/cleaned_sales.csv` from `02_preprocessing.ipynb`; run this notebook from the repository root after generating that CSV. It contains the fixed chronological `DataSplit` and 41 numeric model features. The preprocessing and category encoders were fitted using training rows only.

`ClosePrice` is the target. `CloseDate` and `DataSplit` are audit fields, not predictors. R² on the April 2026 test month is the headline comparison. MAE, RMSE, the train-to-test R² gap, and errors by price band help explain model behavior. The model settings below are fixed in advance; the test set is not used to tune them.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

DATA_PATH = Path('csv/cleaned_sales.csv')
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        'Run 02_preprocessing.ipynb from the repository root to create csv/cleaned_sales.csv.'
    )

## 1. Reuse the Week 3 chronological split

These checks mirror the Week 4 notebook. Every remaining predictor must be numeric and finite, and every training sale must precede the test month.

In [2]:
data = pd.read_csv(DATA_PATH, parse_dates=['CloseDate'])
metadata_columns = ['CloseDate', 'ClosePrice', 'DataSplit']
missing_columns = set(metadata_columns) - set(data.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')
if set(data['DataSplit']) != {'train', 'test'}:
    raise ValueError('DataSplit must contain exactly train and test.')

feature_columns = [column for column in data.columns if column not in metadata_columns]
if not feature_columns:
    raise ValueError('No model features found.')
non_numeric = [column for column in feature_columns
               if not pd.api.types.is_numeric_dtype(data[column])]
if non_numeric:
    raise ValueError(f'Model features must be numeric: {non_numeric}')

X = data.loc[:, feature_columns]
y = pd.to_numeric(data['ClosePrice'], errors='raise')
if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError('Model features must all be finite.')
if not np.isfinite(y.to_numpy(dtype=float)).all() or not y.gt(0).all():
    raise ValueError('ClosePrice labels must be finite and positive.')
if data['CloseDate'].isna().any():
    raise ValueError('Every row needs a valid CloseDate.')

train_mask = data['DataSplit'].eq('train')
test_mask = data['DataSplit'].eq('test')
if data.loc[train_mask, 'CloseDate'].max() >= data.loc[test_mask, 'CloseDate'].min():
    raise ValueError('Training dates must precede test dates.')

X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
split_audit = data.groupby('DataSplit', sort=False).agg(
    rows=('ClosePrice', 'size'),
    first_close=('CloseDate', 'min'),
    last_close=('CloseDate', 'max'),
)
print(f'Model features: {len(feature_columns)}')
display(split_audit)

Model features: 41


,rows,first_close,last_close
DataSplit,,,
train,130274,2025-04-01,2026-03-31
test,12470,2026-04-01,2026-04-30


## 2. Fit three regressors

The single tree can learn nonlinear splits and interactions, but one tree can fit training-specific details. Limiting its depth and requiring at least 20 sales per leaf reduces that risk. The forest averages 100 varied trees, which typically lowers variance; its depth, minimum leaf size, and four-worker limit keep training practical on this dataset. Both use `random_state=42` for reproducibility. The linear baseline is refitted here rather than copied from a prior result, so all three scores use identical rows and features.

In [3]:
models = {
    'Linear regression (Week 4 baseline)': LinearRegression(),
    'Decision tree': DecisionTreeRegressor(
        max_depth=16, min_samples_leaf=20, random_state=42
    ),
    'Random forest': RandomForestRegressor(
        n_estimators=100, max_depth=20, min_samples_leaf=5,
        max_features=0.8, n_jobs=4, random_state=42
    ),
}

rows = []
test_predictions = {}
for name, model in models.items():
    started = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - started
    predicted_train = model.predict(X_train)
    predicted_test = model.predict(X_test)
    test_predictions[name] = predicted_test
    rows.append({
        'Model': name,
        'Train R²': r2_score(y_train, predicted_train),
        'Test R²': r2_score(y_test, predicted_test),
        'Test MAE ($)': mean_absolute_error(y_test, predicted_test),
        'Test RMSE ($)': np.sqrt(mean_squared_error(y_test, predicted_test)),
        'Negative test predictions': int(np.sum(predicted_test < 0)),
        'Fit time (s)': fit_seconds,
    })
    print(f'Fitted {name} in {fit_seconds:.1f} s')

comparison = pd.DataFrame(rows).set_index('Model')
baseline_r2 = comparison.loc['Linear regression (Week 4 baseline)', 'Test R²']
comparison.insert(2, 'Δ R² vs baseline', comparison['Test R²'] - baseline_r2)
comparison.insert(3, 'Train − test R²', comparison['Train R²'] - comparison['Test R²'])
display(comparison.round({
    'Train R²': 4, 'Test R²': 4, 'Δ R² vs baseline': 4,
    'Train − test R²': 4, 'Test MAE ($)': 0,
    'Test RMSE ($)': 0, 'Fit time (s)': 1,
}))

Fitted Linear regression (Week 4 baseline) in 0.2 s
Fitted Decision tree in 2.2 s
Fitted Random forest in 50.9 s


,Train R²,Test R²,Δ R² vs baseline,Train − test R²,Test MAE ($),Test RMSE ($),Negative test predictions,Fit time (s)
Model,,,,,,,,
Linear regression (Week 4 baseline),0.6584,0.6632,0.0000,-0.0047,389746.0,786908.0,623,0.2
Decision tree,0.8521,0.7867,0.1235,0.0654,233533.0,626181.0,0,2.2
Random forest,0.9209,0.8366,0.1734,0.0843,194531.0,548057.0,0,50.9


## 3. Check where errors occur

The following price bands are defined from actual test prices for **evaluation only**; they are never model inputs. Mean prediction error is prediction minus actual, so a negative value indicates underprediction.

In [4]:
price_band = pd.cut(
    y_test,
    bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
    labels=['<$500k', '$500k–$1m', '$1m–$2m', '$2m+'],
    right=False,
)
band_rows = []
for name, predicted in test_predictions.items():
    errors = pd.DataFrame({
        'price_band': price_band.reset_index(drop=True),
        'absolute_error': np.abs(predicted - y_test.to_numpy()),
        'signed_error': predicted - y_test.to_numpy(),
    })
    by_band = errors.groupby('price_band', observed=True).agg(
        sales=('absolute_error', 'size'),
        mae_dollars=('absolute_error', 'mean'),
        mean_error_dollars=('signed_error', 'mean'),
    )
    for band, values in by_band.iterrows():
        band_rows.append({
            'Model': name, 'Actual close-price band': band,
            'Sales': int(values['sales']),
            'MAE ($)': round(values['mae_dollars']),
            'Mean prediction error ($)': round(values['mean_error_dollars']),
        })
band_comparison = pd.DataFrame(band_rows)
display(band_comparison)

,Model,Actual close-price band,Sales,MAE ($),Mean prediction error ($)
0,Linear regression (Week 4 baseline),<$500k,1745,345421,-228001
1,Linear regression (Week 4 baseline),$500k–$1m,5158,264547,66583
2,Linear regression (Week 4 baseline),$1m–$2m,3794,344596,219252
3,Linear regression (Week 4 baseline),$2m+,1773,894218,-498216
4,Decision tree,<$500k,1745,69590,42577
5,Decision tree,$500k–$1m,5158,86361,25266
6,Decision tree,$1m–$2m,3794,214267,6804
7,Decision tree,$2m+,1773,864267,-320541
8,Random forest,<$500k,1745,60890,40847
9,Random forest,$500k–$1m,5158,73409,24784


## 4. Interpretation

On the same 12,470 April 2026 test sales, the **linear baseline scores R² = 0.6632**, the **decision tree scores 0.7867** (**+0.1235** R² points), and the **random forest scores 0.8366** (**+0.1734** points). The forest also has the lowest test MAE ($194,531) and RMSE ($548,057), compared with $389,746 and $786,908 for the linear baseline.

- **Decision tree:** Its nonlinear splits improve test fit and keep predictions positive (zero negative prices versus 623 from linear regression). It is quick to fit, but a single tree gives stepwise predictions and is sensitive to the training sample. Its R² falls from 0.8521 on training to 0.7867 on test, showing some overfitting even with depth and leaf-size limits.
- **Random forest:** Averaging trees produces the strongest test score here and also avoids negative prices. It takes about 51 seconds to fit in this run, versus about 2 seconds for the single tree. Its training R² of 0.9209 still exceeds its test R² by 0.0843, so the ensemble does not eliminate overfitting.
- **High-price weakness:** For sales at $2 million or more, the forest's MAE is $704,448 and its mean prediction error is −$284,362; the tree is also biased low in this band. Tree leaves average training labels, so neither tree model can extrapolate beyond the prices seen in training. The forest tends to overpredict the lowest band by about $40,847 on average.

These scores describe **closed MLS sales**, not off-market homes. The fixed settings are an illustrative comparison, not a hyperparameter search. The repository's separate production valuation evaluation uses different features and dates, so its scores are not the Week 4 baseline for this notebook.